<a href="https://colab.research.google.com/github/rachidahebie-cloud/churnradar/blob/main/churnradar.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 1. Analyse du Churn Client — Telco Customer Churn

**Auteur :** Tié Rachida HEBIE

**Dataset :** Telco Customer Churn (IBM Sample Dataset)
**Source :** Kaggle — https://www.kaggle.com/datasets/blastchar/telco-customer-churn

## Introduction

La rétention client est un enjeu majeur pour les entreprises de télécommunications,
où le coût d'acquisition d'un nouveau client est généralement bien plus élevé que
celui de la fidélisation d'un client existant. Ce projet analyse un jeu de données
de 7 043 clients d'une entreprise de télécommunications fictive, incluant leurs
caractéristiques démographiques, les services souscrits, et leur statut de
désabonnement (churn). L'objectif est d'identifier les facteurs associés au churn
afin de fournir des pistes d'action concrètes pour améliorer la rétention.

##2. Problème et objectifs analytiques

**Question de recherche principale :**
Quels sont les facteurs qui influencent le plus la probabilité qu'un client se
désabonne (churn), et comment ces facteurs peuvent-ils orienter une stratégie
de rétention ?

**Type de problème analytique :**
Ce projet combine une analyse exploratoire (EDA) descriptive et relationnelle,
avec un volet optionnel de classification supervisée (prédiction binaire du churn :
Yes/No).

**Sous-questions :**
- Quelles variables catégorielles (type de contrat, méthode de paiement, services
  souscrits) sont les plus associées au churn ?
- Le profil démographique (senior ou non) influence-t-il le taux de désabonnement ?
- L'ancienneté du client (tenure) et les montants facturés (MonthlyCharges,
  TotalCharges) sont-ils liés à la probabilité de churn ?

**Pertinence opérationnelle :**
Les résultats de cette analyse peuvent permettre à l'entreprise de cibler des
actions de rétention (offres, accompagnement) vers les segments de clients les
plus à risque, plutôt que d'appliquer une stratégie générique coûteuse et peu
efficace.

## 3. Chargement des données et aperçu initial

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

SEED = 42
np.random.seed(SEED)
sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)

DATA_PATH = "WA_Fn-UseC_-Telco-Customer-Churn.csv"

In [ ]:
df = pd.read_csv(DATA_PATH)

print("Dimensions :", df.shape)
print("\nPremières lignes :")
print(df.head())
print("\nTypes des variables :")
print(df.dtypes)

print("\nRépartition de la variable cible :")
print(df["Churn"].value_counts())
print(df["Churn"].value_counts(normalize=True).round(4))

## 4. Diagnostic de la qualité des données

In [ ]:
print("Valeurs manquantes (NaN) au total :", df.isna().sum().sum())
print("Lignes dupliquées :", df.duplicated().sum())
print("customerID uniques :", df["customerID"].is_unique)

# Valeurs vides cachées (chaînes d'espaces) : invisibles pour isna()
print("\nValeurs vides cachées :")
for col in df.select_dtypes(include="object").columns:
    n = (df[col].str.strip() == "").sum()
    if n > 0:
        print(f"  {col} : {n}")

print("\nClients avec tenure = 0 :", (df["tenure"] == 0).sum())

# Modalités de chaque variable catégorielle (repérer les incohérences)
categorical_cols = [
    col for col in df.select_dtypes(include="object").columns
    if col not in ["customerID", "TotalCharges", "Churn"]
]
for col in categorical_cols:
    print(f"{col} -> {sorted(df[col].unique())}")

Le diagnostic ne détecte aucune valeur manquante (NaN), aucun doublon et des identifiants clients tous uniques. Cependant, la colonne TotalCharges contient 11 valeurs vides cachées (des espaces), invisibles pour isna() : elles correspondent exactement aux 11 clients dont tenure (l'ancienneté) est de 0 mois. Les modalités des variables catégorielles sont cohérentes

## 5. Nettoyage et prétraitement

In [ ]:
# 1. Doublons
nb_avant = len(df)
df = df.drop_duplicates()
print("Doublons supprimés :", nb_avant - len(df))

# 2. Conversion de TotalCharges en nombre
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")

# 3. Cases vides de TotalCharges
lignes_vides = df["TotalCharges"].isna()
print("Cases vides dans TotalCharges :", lignes_vides.sum())
print("Toutes ont tenure = 0 :", (df.loc[lignes_vides, "tenure"] == 0).all())
df.loc[lignes_vides, "TotalCharges"] = 0
print("Cases vides restantes :", df["TotalCharges"].isna().sum())

# 4. Valeurs absurdes
print("tenure hors de 0 à 72 :", ((df["tenure"] < 0) | (df["tenure"] > 72)).sum())
print("MonthlyCharges <= 0 :", (df["MonthlyCharges"] <= 0).sum())

cols_net = ["OnlineSecurity", "OnlineBackup", "DeviceProtection",
            "TechSupport", "StreamingTV", "StreamingMovies"]
sans_internet = df["InternetService"] == "No"
erreurs = (df.loc[sans_internet, cols_net] != "No internet service").any(axis=1)
print("Incohérences sur les services internet :", erreurs.sum())

# 5. Outliers (global)
for col in ["tenure", "MonthlyCharges", "TotalCharges"]:
    q1 = df[col].quantile(0.25)
    q3 = df[col].quantile(0.75)
    iqr = q3 - q1
    nb_outliers = ((df[col] < q1 - 1.5 * iqr) | (df[col] > q3 + 1.5 * iqr)).sum()
    print(col, ":", nb_outliers, "outliers (global)")

Les contrôles de cohérence ne révèlent aucune valeur absurde : l'ancienneté reste entre 0 et 72 mois, les charges mensuelles sont toutes positives, et aucun client sans internet n'a de service internet. Le calcul des valeurs extrêmes (méthode IQR) sur l'ensemble du dataset n'en détecte aucune pour tenure, MonthlyCharges et TotalCharges.

Le dataset contient 11 valeurs maquantes au niveau de TotalCharges.

Les 11 valeurs manquantes correspondent toutes à des clients avec tenure = 0, c'est-à-dire des clients nouvellement inscrits n'ayant pas encore été facturés. Remplacer par 0 est donc logiquement justifié plutôt que d'insérer la moyenne ou de supprimer ces lignes.

### 5.1 Création de nouvelles variables (feature engineering)

In [ ]:
# 1. Cible en 0/1
df["Churn_bin"] = (df["Churn"] == "Yes").astype(int)

# 2. Tranches d'ancienneté
df["tenure_group"] = pd.cut(
    df["tenure"],
    bins=[-1, 12, 24, 48, 72],
    labels=["0-12", "13-24", "25-48", "49-72"]
)

# 3. Nombre de services optionnels
df["nb_services"] = (df[cols_net] == "Yes").sum(axis=1)

# 4. Paiement automatique (1) ou manuel (0)
df["auto_payment"] = df["PaymentMethod"].str.contains("automatic").astype(int)

# Vérification : taux de churn (%) pour chaque nouvelle variable
print(df.groupby("tenure_group", observed=True)["Churn_bin"].mean().mul(100).round(2))
print()
print(df.groupby("nb_services")["Churn_bin"].mean().mul(100).round(2))
print()
print(df.groupby("auto_payment")["Churn_bin"].mean().mul(100).round(2))

In [ ]:
print(df.groupby(["InternetService", "nb_services"])["Churn_bin"].mean().mul(100).round(2))

Le taux de churn baisse nettement quand l'ancienneté augmente : il passe de 47,44 % la première année (0-12 mois) à 9,51 % après quatre ans (49-72 mois). Les clients qui paient par prélèvement ou carte automatique résilient aussi moins (15,98 %) que ceux qui paient manuellement (34,67 %).

Le churn diminue également quand le nombre de services optionnels augmente. Le groupe à 0 service (21,41 %) ne suit pas cette tendance parce qu'il mélange deux profils : les clients sans internet, qui ne peuvent pas souscrire ces services et résilient peu (7,40 %), et les clients avec internet sans aucune option, qui résilient beaucoup (41,16 % en DSL et 60,40 % en fibre optique). En séparant selon le type d'internet, la baisse du churn est régulière dans les deux cas, et à nombre de services égal la fibre optique résilie plus que le DSL. Il faut donc lire cette variable en la croisant avec InternetService.

## 6. Analyse exploratoire (EDA)
### 6.1 Statistiques descriptives

In [ ]:
print(df.describe())

### 6.2 Analyse des variables catégorielles

In [ ]:
for col in categorical_cols:
    print("\n" + "=" * 50)
    print(f"Variable : {col}")

    print("\nRépartition :")
    print(df[col].value_counts())

    print("\nPourcentage :")
    print((df[col].value_counts(normalize=True) * 100).round(2))

    print("\nTaux de churn par modalité :")
    print(
        (pd.crosstab(
            df[col],
            df["Churn"],
            normalize="index"
        ) * 100).round(2)
    )

Le churn semble particulièrement élevé chez les clients ayant des contrats mensuels, utilisant l'Electronic check, ayant une connexion Fiber optic et ne disposant pas de certains services complémentaires comme OnlineSecurity ou TechSupport.

### 6.3 Analyse des variables non catégorielles
#### SeniorCitizen (Senior = 1, non-senior = 0)

In [ ]:
print(df["SeniorCitizen"].value_counts())

print(df["SeniorCitizen"].value_counts(normalize=True) * 100)

print(
    pd.crosstab(
        df["SeniorCitizen"],
        df["Churn"],
        normalize="index"
    ) * 100
)

Les seniors sont davantage susceptibles de se désabonner

### tenure (durée d'abonnement d'un client)

In [ ]:
print(df.groupby("Churn")["tenure"].describe())

Les clients qui se désabonnent ont une ancienneté nettement plus faible (médiane environ 10 mois) que ceux qui restent (environ 38 mois). Les nouveaux clients sont donc les plus à risque de churn, ce qui pourrait suggérer un enjeu d'accompagnement dans les premiers mois de la relation client.

### MonthlyCharges (charge mensuelle de chaque client )

In [ ]:
print(df.groupby("Churn")["MonthlyCharges"].describe())

Les clients qui restent abonnés paient une charge mensuelle moins élevée que ceux qui se désabonnent. On peut donc supposer que les clients dont la charge mensuelle est plus élevée sont davantage susceptibles de résilier.

### TotalCharges (montant total payé depuis l'abonnement d'un client)

In [ ]:
print(df.groupby("Churn")["TotalCharges"].describe())

Les clients qui se désabonnent ont une charge totale nettement plus faible que ceux qui restent, avec une médiane de 703,55 dollars contre 1 679,53 dollars. Cette différence peut s'expliquer en partie par leur ancienneté plus faible : les clients qui se désabonnent restent généralement moins longtemps dans l'entreprise et ont donc moins de temps pour accumuler des charges.

### 6.4 Visualisations

In [ ]:
sns.histplot(data = df, x="tenure")

plt.title("Visualisation de tenure")
plt.xlabel("tenure")
plt.ylabel("Nombre de clients")

plt.show()

la distribution est bimodale, avec une forte concentration de clients à très faible ancienneté et un second groupe à ancienneté maximale.

In [ ]:
sns.histplot(data = df, x="TotalCharges")

plt.title("Visualisation de TotalCharges")
plt.xlabel("totalCharges")
plt.ylabel("Number of cosumers")

plt.show()

Le graphique montre que la majorité des clients ont un TotalCharges faible, avec une distribution fortement asymétrique à droite : le nombre de clients diminue progressivement à mesure que le montant total augmente. Cette forme s'explique en grande partie par la distribution de tenure observée précédemment : de nombreux clients ayant une faible ancienneté, ils n'ont mécaniquement pas eu le temps d'accumuler des charges élevées, indépendamment de leur charge mensuelle.

In [ ]:
sns.histplot(data = df, x="MonthlyCharges")

plt.title("Visualisation de MonthlyCharges")
plt.xlabel("MonthlyCharges")
plt.ylabel("Number of cosumers")

plt.show()

La distribution de MonthlyCharges est bimodale : un premier pic se situe autour de 20 dollars, puis la majorité des clients se répartit entre 60 et 110 dollars environ. Cette forme s'explique par le type de service internet souscrit, comme le montre le graphique suivant.

In [ ]:
sns.histplot(data = df, x="MonthlyCharges", hue="InternetService")

plt.title("Distribution de MonthlyCharges selon le type d'internet")
plt.xlabel("MonthlyCharges")
plt.ylabel("Number of cosumers")

plt.show()

Ce graphique confirme parfaitement l'hypothèse formulée précédemment sur l'origine de la distribution bimodale de MonthlyCharges. On voit clairement que le premier pic autour de 20 dollars correspond presque exclusivement aux clients sans service internet, en vert, ce qui est logique puisqu'ils ne paient que le téléphone. Le second groupe, situé entre 70 et 100 dollars, est majoritairement composé de clients ayant la fibre optique, en orange, dont les charges mensuelles sont nettement plus élevées à cause du coût plus important de ce service. Les clients avec une connexion DSL, en bleu, se situent dans une zone intermédiaire, avec des charges mensuelles généralement comprises entre 40 et 70 dollars. On peut donc conclure que la bimodalité observée n'est pas due au hasard mais reflète directement le type de service internet souscrit par les clients, chaque groupe technologique ayant une structure de prix propre. Ce résultat est particulièrement intéressant si on le met en lien avec les taux de churn calculés précédemment, puisque les clients fibre optique, qui paient le plus cher, sont aussi ceux qui présentent le taux de désabonnement le plus élevé avec 41,89 pourcent, contre 18,96 pourcent pour les clients DSL et seulement 7,40 pourcent pour les clients sans internet. Cela suggère un lien possible entre le niveau de charge mensuelle et la probabilité de churn, les clients payant plus cher étant aussi les plus susceptibles de se désabonner.

Boxplot 1 : tenure selon Churn

In [ ]:
sns.boxplot(data=df, x="Churn", y="tenure")
plt.title("Boxplot de tenure selon Churn")
plt.show()

Le boxplot confirme visuellement l'écart déjà observé dans les statistiques descriptives : la médiane d'ancienneté des clients qui résilient (10 mois) est nettement inférieure à celle des clients qui restent (38 mois), avec un chevauchement limité entre les deux distributions. Les clients qui restent affichent une dispersion plus large, incluant aussi bien des clients récents que très anciens, tandis que les clients qui résilient se concentrent majoritairement sur de faibles anciennetés. Quelques clients résilient tout de même après une longue ancienneté (jusqu'à 72 mois) : c'est rare, mais cela mérite d'être signalé comme exception au profil général.

Boxplot 2 : MonthlyCharges selon Churn

In [ ]:
sns.boxplot(data=df, x="Churn", y="MonthlyCharges")
plt.title("Boxplot de MonthlyCharges selon Churn")
plt.show()

On observe que les clients qui se désabonnent ont des charges mensuelles plus élevées (médiane d'environ 80 dollars) que ceux qui restent (médiane d'environ 64 dollars), ce qui confirme les statistiques calculées précédemment (moyennes de 74 dollars contre 61 dollars). Contrairement aux boxplots de tenure et TotalCharges, c'est ici le groupe "Yes" qui est positionné plus haut. Le chevauchement entre les deux boîtes reste important : la charge mensuelle seule ne permet pas de distinguer parfaitement les deux groupes, car de nombreux clients avec une charge élevée restent abonnés. La boîte "No" est plus étalée que la boîte "Yes", ce qui montre que les clients fidèles ont des profils de charges plus variés, alors que les clients qui résilient se concentrent davantage dans le haut de la fourchette de prix. Aucune valeur extrême n'apparaît sur cette variable, contrairement à TotalCharges.

Boxplot 3 : TotalCharges selon Churn

In [ ]:
sns.boxplot(data=df, x="Churn", y="TotalCharges")
plt.title("Boxplot de TotalCharges selon Churn")
plt.show()

Le boxplot confirme que les clients ayant résilié ont un TotalCharges médian nettement plus faible (environ 704 dollars) que ceux qui sont restés (environ 1 680 dollars), ce qui est cohérent avec leur ancienneté généralement plus courte. La boîte des clients restés est également plus étalée, ce qui reflète une plus grande diversité de profils (nouveaux et anciens clients confondus). On observe cependant de nombreuses valeurs extrêmes au-dessus du groupe "Yes", jusqu'à environ 8 700 dollars. Il s'agit de valeurs extrêmes au sein de ce groupe (au-delà de sa moustache), et non du dataset entier : ces clients ont accumulé une charge totale élevée, donc probablement une ancienneté ou des charges mensuelles importantes, mais ont tout de même résilié. Ce sous-groupe s'écarte du profil typique du churner et pourrait mériter une investigation plus poussée (hausse tarifaire ou problème de service avant le départ), information non disponible dans ce dataset.

KDE plot : distribution des variables numériques selon Churn

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, col in zip(axes, ["tenure", "MonthlyCharges", "TotalCharges"]):
    sns.kdeplot(data=df, x=col, hue="Churn", fill=True, common_norm=False, ax=ax)
    ax.set_title(f"Distribution de {col} selon Churn")
plt.tight_layout()
plt.show()

Les courbes de densité confirment les écarts observés dans les statistiques. Pour tenure, les clients qui se désabonnent sont fortement concentrés sur les tout premiers mois (pic proche de 0), alors que les clients qui restent sont répartis sur toute la durée, avec une remontée vers 70 mois. Pour MonthlyCharges, les clients qui restent ont un pic marqué autour de 20 dollars (clients sans internet), alors que les clients qui partent sont surtout situés entre 70 et 105 dollars. Pour TotalCharges, les deux groupes sont concentrés sur les faibles montants, mais les clients qui partent le plus sont plus nombreux, ce qui s'explique par leur ancienneté plus faible.

Scatter : tenure vs MonthlyCharges selon Churn

In [ ]:
plt.figure(figsize=(8, 5))
sns.scatterplot(data=df, x="tenure", y="MonthlyCharges", hue="Churn",
                alpha=0.5, palette={"No": "#66b3ff", "Yes": "#ff6666"})
plt.title("Ancienneté vs charges mensuelles selon Churn")
plt.xlabel("Ancienneté (mois)")
plt.ylabel("Charges mensuelles")
plt.show()

Le nuage de points montre que les clients qui se désabonnent (en rouge) se situent surtout à gauche du graphique, donc avec une faible ancienneté, et dans la partie haute, avec des charges mensuelles élevées (environ 70 à 105 dollars). Au-delà de 60 mois, presque tous les clients sont restés, quel que soit leur tarif. La bande autour de 20 dollars est presque entièrement bleue : ces clients, sans internet, résilient très peu. Le risque de churn est donc maximal quand une faible ancienneté se combine à une charge mensuelle élevée.

Affichage des variable catégorielle en barres plots horizontales

In [ ]:
variables_peu_discriminantes = ["gender", "PhoneService", "MultipleLines", "StreamingTV", "StreamingMovies"]

for col in variables_peu_discriminantes:
    churn_rate = (
        pd.crosstab(df[col], df["Churn"], normalize="index")["Yes"] * 100
    ).sort_values()

    plt.figure(figsize=(7, 3))
    churn_rate.plot(kind="barh", color="gray")
    plt.title(f"Taux de churn (%) par {col}")
    plt.xlabel("Taux de churn (%)")
    plt.ylabel(col)
    plt.tight_layout()
    plt.show()

Certaines caractéristiques montrent peu ou pas de lien avec le churn. Le genre (gender) affiche un taux de désabonnement quasi identique entre hommes et femmes (26,9% vs 26,2%), de même que la possession d'un service téléphonique (PhoneService) ou de plusieurs lignes (MultipleLines), avec des écarts inférieurs à 4 points. Les services de streaming (StreamingTV, StreamingMovies) présentent un écart un peu plus visible mais qui reste modeste (environ 3 à 4 points entre "Yes" et "No"), bien moins marqué que pour les variables contractuelles ou techniques (Contract, InternetService, OnlineSecurity). Cela suggère que ces caractéristiques ne constituent pas des facteurs déterminants du désabonnement, contrairement au type de contrat, à la méthode de paiement ou à la présence de services de sécurité/support technique.

In [ ]:
variables_discriminantes = ["Contract", "PaymentMethod", "InternetService", "OnlineSecurity",
                            "TechSupport", "Dependents", "Partner", "PaperlessBilling",
                            "OnlineBackup", "DeviceProtection", "SeniorCitizen"]

for col in variables_discriminantes:
    churn_rate = (
        pd.crosstab(df[col], df["Churn"], normalize="index")["Yes"] * 100
    ).sort_values()

    plt.figure(figsize=(7, 4))
    churn_rate.plot(kind="barh", color="steelblue")
    plt.title(f"Taux de churn (%) par {col}")
    plt.xlabel("Taux de churn (%)")
    plt.ylabel(col)
    plt.tight_layout()
    plt.show()

Ces graphiques confirment visuellement les tendances identifiées dans l'analyse précédente. Les onze variables retenues (Contract, PaymentMethod, InternetService, OnlineSecurity, TechSupport, Dependents, Partner, PaperlessBilling, OnlineBackup, DeviceProtection, SeniorCitizen) présentent toutes un écart de taux de churn d'au moins 13 points entre leurs modalités extrêmes, contre moins de 4 points pour gender et PhoneService, et environ 4 points pour MultipleLines, StreamingTV et StreamingMovies, ce qui justifie leur sélection comme facteurs discriminants.

Le type de contrat reste le facteur le plus marquant : le churn passe de 2,8% pour un engagement de deux ans à 42,7% pour un contrat mensuel, soit un écart de près de 40 points. La méthode de paiement Electronic check se détache également nettement (45,3% de churn, contre 15 à 19% pour les autres méthodes), tout comme la fibre optique (41,9% contre 7,4% pour les clients sans internet). Enfin, l'absence de services de sécurité ou de support technique (OnlineSecurity, TechSupport) double quasiment le taux de résiliation par rapport aux clients qui en bénéficient.

Ces résultats dessinent un profil cohérent du client à risque : peu engagé contractuellement, non accompagné par des services de protection, et utilisant un mode de paiement moins automatisé. Cette lecture croisée suggère que le churn n'est pas dû à un facteur isolé, mais à une combinaison de signaux liés à l'engagement et à l'accompagnement du client.

heatmap de corrélation

In [ ]:
cols_corr = ["tenure", "MonthlyCharges", "TotalCharges", "SeniorCitizen",
             "nb_services", "auto_payment", "Churn_bin"]

plt.figure(figsize=(8, 6))
sns.heatmap(df[cols_corr].corr(), annot=True, cmap="coolwarm", center=0, fmt=".2f")
plt.title("Matrice de corrélation des variables numériques et de la cible")
plt.show()

La matrice de corrélation révèle une relation forte entre tenure et TotalCharges (0,83), ce qui s'explique mécaniquement : plus un client reste longtemps abonné, plus il accumule de facturations, indépendamment de tout autre facteur. La corrélation entre MonthlyCharges et TotalCharges est modérée (0,65) : un client qui paie cher chaque mois tend à avoir un total plus élevé, mais cette relation est moins automatique que celle avec tenure, car elle dépend aussi de la durée d'abonnement.

En revanche, tenure et MonthlyCharges sont très peu corrélées (0,25), ce qui indique que l'ancienneté d'un client n'est pas liée à ce qu'il paie chaque mois : un nouveau client peut très bien souscrire une offre coûteuse (fibre + options), tandis qu'un client fidèle peut conserver un forfait basique. Enfin, SeniorCitizen n'affiche quasiment aucune corrélation avec les autres variables numériques (0,02 à 0,22), suggérant que le statut senior influence le churn par d'autres mécanismes que ceux mesurés ici (comportementaux ou liés au type de services souscrits, plutôt que financiers).

Il est important de noter que ces corrélations ne mesurent que des relations linéaires et ne permettent pas de conclure à une causalité. La forte corrélation entre tenure et TotalCharges reflète notamment un lien de calcul plutôt qu'un effet explicatif à interpréter d'un point de vue business.

La dernière ligne de la matrice montre le lien de chaque variable avec le churn. L'ancienneté (tenure) est la plus fortement liée, de façon négative (-0,35) : plus un client est ancien, moins il se désabonne. Le paiement automatique (-0,21) et la charge totale (-0,20) sont aussi négativement liés au churn, tandis que la charge mensuelle (0,19) et le statut senior (0,15) y sont légèrement liés de façon positive. Le nombre de services (-0,09) a une corrélation linéaire faible avec le churn, car son effet dépend du type d'internet (voir section 5.1). Toutes ces corrélations restent faibles à modérées (en valeur absolue, 0,35 au maximum) : le churn dépend surtout de combinaisons de facteurs, comme contrat mensuel et fibre optique, qu'une corrélation linéaire simple ne capte pas.

On note aussi que nb_services est fortement corrélé à TotalCharges (0,74) et à MonthlyCharges (0,72) : un client qui souscrit plus de services paie logiquement plus cher chaque mois et accumule un total plus élevé.

Graphiques des variables dérivées

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, col in zip(axes, ["tenure_group", "nb_services", "auto_payment"]):
    taux = df.groupby(col, observed=True)["Churn_bin"].mean() * 100
    taux.plot(kind="bar", ax=ax, color="steelblue")
    ax.set_title(f"Taux de churn (%) par {col}")
    ax.set_ylabel("Taux de churn (%)")
    ax.tick_params(axis="x", rotation=0)
plt.tight_layout()
plt.show()

Ces graphiques confirment les résultats du feature engineering. Le taux de churn diminue nettement avec l'ancienneté, de 47,44 % la première année à 9,51 % après quatre ans. Il baisse aussi quand le nombre de services augmente, à l'exception du groupe à 0 service, qui mélange les clients sans internet et les clients avec internet sans option (voir section 5.1). Enfin, les clients qui paient automatiquement résilient deux fois moins (15,98 %) que ceux qui paient manuellement (34,67 %).

Camembert

In [ ]:
churn_counts = df["Churn"].value_counts()

plt.figure(figsize=(5, 5))
churn_counts.plot(kind="pie", autopct="%1.1f%%", colors=["#66b3ff", "#ff6666"])
plt.title("Répartition globale du Churn")
plt.ylabel("")
plt.show()

Le camembert montre que 26,5 % des clients se sont désabonnés contre 73,5 % qui sont restés.

## 7. Manipulation et agrégation des données

groupby avec plusieurs agrégations

In [ ]:
resume_tenure = df.groupby("tenure_group", observed=True).agg(
    nb_clients=("Churn_bin", "size"),
    taux_churn=("Churn_bin", "mean"),
    charge_mensuelle_moy=("MonthlyCharges", "mean"),
    charge_totale_moy=("TotalCharges", "mean")
)
resume_tenure["taux_churn"] = (resume_tenure["taux_churn"] * 100).round(2)
resume_tenure["charge_mensuelle_moy"] = resume_tenure["charge_mensuelle_moy"].round(2)
resume_tenure["charge_totale_moy"] = resume_tenure["charge_totale_moy"].round(2)
print(resume_tenure)

Le taux de churn diminue nettement avec l'ancienneté, de 47,44 % pour les clients de 0 à 12 mois à 9,51 % pour ceux de plus de 48 mois. La charge mensuelle moyenne augmente en sens inverse (de 56,10 à 73,95 dollars) : les clients les plus anciens ne sont donc pas ceux qui paient le moins. Leur fidélité ne s'explique pas par un tarif plus bas. La charge totale moyenne augmente fortement avec l'ancienneté (de 275,23 à 4 685,51 dollars), ce qui est logique puisqu'elle cumule les paiements dans le temps.

filtrage et tri

In [ ]:
segments = df.groupby(["Contract", "InternetService", "PaymentMethod"]).agg(
    nb_clients=("Churn_bin", "size"),
    taux_churn=("Churn_bin", "mean")
)
segments["taux_churn"] = (segments["taux_churn"] * 100).round(2)

# Filtrage : segments d'au moins 50 clients
segments = segments[segments["nb_clients"] >= 50]

# Tri : du plus risqué au moins risqué
segments = segments.sort_values("taux_churn", ascending=False)

print(segments.head(10))

Parmi les segments d'au moins 50 clients, le plus à risque combine contrat mensuel, fibre optique et paiement par Electronic check : 60,37 % de churn sur 1 307 clients. Les dix premiers segments du classement sont presque tous en contrat mensuel, et les segments en fibre optique y sont au-dessus des segments DSL. Pour un même contrat et un même type de connexion, le paiement par Electronic check est le plus risqué : en contrat mensuel avec fibre, il atteint 60,37 % contre 41,64 % à 45,57 % pour les paiements automatiques. Ce classement affine le segment « contrat mensuel + fibre » identifié plus bas en montrant que le mode de paiement aggrave encore le risque.

In [ ]:
pivot_churn = df.pivot_table(
    index="Contract",
    columns="InternetService",
    values="Churn",
    aggfunc=lambda x: (x == "Yes").mean() * 100
)

print(pivot_churn.round(2))

Ce tableau croisé révèle un effet d'interaction important entre le type de contrat et le type de connexion internet. Le taux de churn le plus élevé du dataset (54,61%) concerne les clients combinant un contrat mensuel et une connexion fibre optique — soit plus du double du taux observé pour les clients en contrat mensuel avec DSL (32,22%).

Plus intéressant encore, l'effet de la fibre optique sur le churn varie fortement selon l'engagement contractuel : pour un contrat de deux ans, la fibre ne fait passer le churn que de 1,91% à 7,23% (un écart limité), tandis que pour un contrat mensuel, cet écart atteint plus de 22 points. Cela suggère que ce n'est pas la fibre optique en elle-même qui pose problème, mais plutôt la combinaison d'un service coûteux avec un engagement faible, qui laisse au client une grande liberté de résiliation dès qu'il est insatisfait (prix, qualité de service, etc.).

D'un point de vue opérationnel, cela identifie un segment de clients hautement prioritaire pour une stratégie de rétention ciblée : les abonnés fibre optique en contrat mensuel, qui représentent le profil le plus à risque du dataset.

On isole maintenant ce segment à haut risque identifié : contrat mensuel + fibre optique.

In [ ]:
clients_haut_risque = df[
    (df["Contract"] == "Month-to-month") & (df["InternetService"] == "Fiber optic")
]

print("Nombre de clients dans ce segment :", clients_haut_risque.shape[0])
print("Pourcentage du total :", round(clients_haut_risque.shape[0] / df.shape[0] * 100, 2), "%")

print("\nTaux de churn dans ce segment :")
print(clients_haut_risque["Churn"].value_counts(normalize=True) * 100)

print("\nProfil moyen de ce segment :")
print(clients_haut_risque[["tenure", "MonthlyCharges", "TotalCharges"]].describe())

Le segment des clients en contrat mensuel avec fibre optique représente 2 128 clients, soit 30,2 % du dataset, une part loin d'être marginale. Le taux de churn y atteint 54,6 %, plus du double de la moyenne générale du dataset (26,5 %), ce qui en fait le profil le plus à risque identifié dans cette analyse.

Le profil moyen de ce segment (ancienneté de 21,5 mois, charges mensuelles de 87 dollars) montre une ancienneté inférieure à la moyenne générale (32 mois) et des charges mensuelles nettement supérieures (65 dollars en moyenne sur l'ensemble du dataset). Ce segment combine donc trois facteurs de risque : faible engagement contractuel, connexion coûteuse et ancienneté réduite.

D'un point de vue opérationnel, ce segment constitue une cible prioritaire pour une stratégie de rétention : représentant près d'un tiers de la clientèle avec un risque de désabonnement démultiplié, une action ciblée (offre de migration vers un contrat engagé, tarif préférentiel, accompagnement renforcé) pourrait avoir un impact significatif sur le taux de churn global de l'entreprise.

## 8. Modélisation supervisée
### 8.1 Préparation des données

In [ ]:
from sklearn.model_selection import train_test_split

X = df.drop(columns=["customerID", "Churn", "Churn_bin", "tenure_group"])
y = df["Churn_bin"]

X = pd.get_dummies(X, drop_first=True)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=SEED)

print("Nombre de variables :", X.shape[1])
print("Train :", X_train.shape, "| Test :", X_test.shape)
print("Churn dans train :", round(y_train.mean() * 100, 2), "%")
print("Churn dans test :", round(y_test.mean() * 100, 2), "%")

### 8.2 Entraînement et évaluation des modèles

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay, roc_auc_score

modeles = {
    "Régression logistique": make_pipeline(
        StandardScaler(),
        LogisticRegression(max_iter=1000, class_weight="balanced", random_state=SEED)),
    "Random Forest": RandomForestClassifier(
        n_estimators=300, max_depth=8, class_weight="balanced",
        random_state=SEED, n_jobs=-1),
}

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for ax, (nom, modele) in zip(axes, modeles.items()):
    modele.fit(X_train, y_train)
    prediction = modele.predict(X_test)
    proba = modele.predict_proba(X_test)[:, 1]

    print("=" * 50)
    print(nom)
    print(classification_report(y_test, prediction, target_names=["Reste", "Churn"]))
    print("ROC-AUC :", round(roc_auc_score(y_test, proba), 3))

    ConfusionMatrixDisplay(confusion_matrix(y_test, prediction),
                           display_labels=["Reste", "Churn"]).plot(ax=ax, colorbar=False)
    ax.set_title(nom)
plt.tight_layout()
plt.show()

Les deux modèles obtiennent des performances similaires, avec un ROC-AUC de 0,841, indiquant une bonne capacité à distinguer les clients susceptibles de churner de ceux qui restent. Le Random Forest obtient toutefois de meilleures performances globales, avec une accuracy de 75 % et un F1-score de 0,63 sur la classe Churn, contre respectivement 74 % et 0,61 pour la régression logistique. La régression logistique présente néanmoins un recall légèrement supérieur sur la classe Churn (78 % contre 77 %), ce qui signifie qu'elle détecte marginalement plus de clients qui quittent réellement l'entreprise.

### 8.3 : courbes ROC et précision-rappel des 2 modèles

In [ ]:
from sklearn.metrics import RocCurveDisplay, PrecisionRecallDisplay

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for nom, modele in modeles.items():
    proba = modele.predict_proba(X_test)[:, 1]
    RocCurveDisplay.from_predictions(y_test, proba, name=nom, ax=axes[0])
    PrecisionRecallDisplay.from_predictions(y_test, proba, name=nom, ax=axes[1])

axes[0].plot([0, 1], [0, 1], "k--", alpha=0.4, label="Hasard")
axes[0].set_title("Courbe ROC (jeu de test)")
axes[0].legend()
axes[1].axhline(y_test.mean(), color="k", ls="--", alpha=0.4, label="Hasard")
axes[1].set_title("Courbe précision-rappel (jeu de test)")
axes[1].legend()
plt.tight_layout()
plt.show()

Les deux courbes ROC sont quasiment superposées (AUC = 0,84 pour les deux) : les modèles distinguent aussi bien les clients qui partent de ceux qui restent. Sur la courbe précision-rappel, le Random Forest est un peu meilleur (AP = 0,65 contre 0,63), surtout pour les faibles rappels, où ses premières prédictions sont plus fiables. Au-delà d'un rappel de 0,4, les deux courbes se confondent. Les deux modèles sont nettement au-dessus du hasard (AUC = 0,5 ; précision = 26,5 %, soit le taux de churn).

### 8.4 : importance des variables pour les 2 modèles

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Régression logistique : valeur absolue des coefficients (variables standardisées)
lr = modeles["Régression logistique"][-1]
coef = pd.Series(lr.coef_[0], index=X_train.columns)
top_lr = coef.abs().sort_values().tail(10).index
coef[top_lr].plot(kind="barh", ax=axes[0],
                  color=["#ff6666" if c > 0 else "#66b3ff" for c in coef[top_lr]])
axes[0].set_title("Régression logistique : 10 plus gros coefficients\n(rouge = augmente le churn, bleu = diminue)")
axes[0].set_xlabel("Coefficient")

# Random Forest : importance par impuretés
rf = modeles["Random Forest"]
imp_rf = pd.Series(rf.feature_importances_, index=X_train.columns).sort_values().tail(10)
imp_rf.plot(kind="barh", ax=axes[1], color="seagreen")
axes[1].set_title("Random Forest : 10 variables les plus importantes")
axes[1].set_xlabel("Importance")

plt.tight_layout()
plt.show()

Les deux modèles s'accordent sur les variables les plus importantes : l'ancienneté (tenure), le type de contrat, la fibre optique et les charges (MonthlyCharges, TotalCharges). Cela confirme l'EDA. Le Random Forest ajoute le paiement par Electronic check, la sécurité en ligne et le support technique.

### 8.5 : comparaison finale

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, average_precision_score

lignes = []
for nom, modele in modeles.items():
    pred = modele.predict(X_test)
    proba = modele.predict_proba(X_test)[:, 1]
    lignes.append({
        "Modèle": nom,
        "Accuracy": accuracy_score(y_test, pred),
        "Précision (Churn)": precision_score(y_test, pred),
        "Recall (Churn)": recall_score(y_test, pred),
        "F1 (Churn)": f1_score(y_test, pred),
        "ROC-AUC": roc_auc_score(y_test, proba),
        "PR-AUC": average_precision_score(y_test, proba),
    })
print(pd.DataFrame(lignes).set_index("Modèle").round(3))

Les écarts sont faibles (au plus 2 points) et le ROC-AUC est identique. Sur un jeu de test d'environ 1 400 clients, de tels écarts ne sont pas assez fiables pour déclarer un modèle meilleur : les deux sont considérés comme équivalents. Le Random Forest est légèrement plus précis (moins de fausses alertes), la régression logistique détecte légèrement plus de futurs partants.

Choix : la régression logistique. Dans une logique de rétention, manquer un client qui part coûte plus cher qu'une offre envoyée à un client qui serait resté, donc le recall sur « Churn » prime. Le modèle est de plus simple et rapide à ré-entraîner, et il indique le sens de l'effet de chaque variable. Le Random Forest reste préférable si l'entreprise veut limiter le coût des offres, car il génère moins de fausses alertes.

## 9. Synthèse et interprétation opérationnelle

### Principaux résultats

L'analyse de 7 043 clients révèle un taux de churn global de 26,5%. Le désabonnement n'est pas réparti uniformément : il se concentre sur des profils précis.

**Facteurs contractuels et financiers.** Le type de contrat est le facteur le plus discriminant : le churn atteint 42,7% en contrat mensuel, contre 11,3% à un an et 2,8% à deux ans. Le mode de paiement par electronic check (45,3%) et la connexion fibre optique (41,9%) sont également associés à un churn élevé. Les clients qui résilient paient en moyenne plus cher chaque mois (74 dollars contre 61 dollars).

**Facteurs liés à l'accompagnement.** L'absence de services de sécurité en ligne ou de support technique multiplie par près de trois le taux de churn (environ 42% contre 15%).

**Facteurs liés à l'ancienneté et au profil.** Les clients qui résilient ont une ancienneté médiane de 10 mois, contre 38 mois pour ceux qui restent : le risque est maximal en début de relation. Les clients seniors (41,7% de churn contre 23,6%), sans partenaire ou sans personnes à charge résilient aussi davantage. À l'inverse, le genre, le service téléphonique et les options de streaming n'ont quasiment pas d'influence.

**Effet d'interaction.** Le tableau croisé Contract x InternetService montre que les facteurs se renforcent mutuellement. Le churn atteint 54,6% pour les clients en contrat mensuel avec fibre optique, alors qu'il n'est que de 7,2% pour la fibre en contrat de deux ans.

**Modélisation.** Deux modèles ont été entraînés pour prédire le churn : une régression logistique et un Random Forest. Ils obtiennent des performances très proches (ROC-AUC de 0,84 pour les deux) et sont donc considérés comme équivalents. Les deux s'accordent sur les variables les plus importantes : l'ancienneté, le type de contrat, la fibre optique et les charges, ce qui confirme les résultats de l'EDA. La régression logistique est retenue : elle détecte 78% des clients qui partent (recall) et elle est facile à interpréter.

### Implications opérationnelles

Le segment "contrat mensuel + fibre optique" regroupe 2 128 clients (30,2% de la base) avec un taux de churn plus de deux fois supérieur à la moyenne. Il constitue la cible prioritaire d'une stratégie de rétention. Quatre pistes d'action se dégagent :

1. **Inciter à l'engagement** : proposer aux clients fibre en contrat mensuel une migration vers un contrat annuel ou de deux ans avec un avantage tarifaire.
2. **Accompagner les nouveaux clients** : mettre en place un suivi renforcé durant les premiers mois, période où le risque de résiliation est le plus élevé.
3. **Promouvoir les services de protection** : proposer la sécurité en ligne et le support technique, associés à un churn nettement plus faible.
4. **Utiliser le modèle pour cibler les offres** : calculer pour chaque client sa probabilité de départ avec la régression logistique et envoyer les offres de rétention en priorité aux clients les plus à risque, plutôt qu'à toute la base.

### Lien avec les objectifs initiaux

Ces résultats répondent à la question de recherche posée en section 2. Les facteurs les plus associés au churn sont le faible engagement contractuel, l'abonnement récent, le type de connexion et l'absence de services d'accompagnement. L'EDA les met en évidence et les modèles les confirment. Ils permettent de cibler des segments précis plutôt que d'appliquer une stratégie de rétention générique.

## 10. Discussion critique et limites

### Limites du dataset
- **Données fictives** : le dataset est un exemple publié par IBM, pas des données réelles d'un opérateur. Les résultats montrent des tendances utiles pour l'analyse, mais on ne peut pas les appliquer directement à une vraie entreprise.
- **Une seule photographie** : les données ne contiennent pas de dates. On ne voit pas l'évolution des clients dans le temps (changement de contrat, hausse de prix, saisonnalité).
- **Informations manquantes** : on ne connaît ni la raison du départ, ni la qualité du service, ni les offres des concurrents, ni les réclamations. On sait *qui* part, mais pas *pourquoi*.
- **Peu de seniors** : ils représentent seulement 16 % des clients (1 142), donc les conclusions sur ce groupe sont moins solides.

### Hypothèses faites
- Les 11 `TotalCharges` vides ont été remplacées par 0, car ces clients ont `tenure = 0` et n'ont pas encore été facturés.
- Les tranches d'ancienneté (0-12, 13-24, 25-48, 49-72) et le seuil de 50 clients par segment sont des choix personnels. Un autre découpage pourrait donner des chiffres un peu différents.
- Les clients sans internet ont été gardés tels quels, bien qu'ils forment un groupe très particulier (7,4 % de churn).

### Biais possibles
- **Association n'est pas causalité** : par exemple, les clients en contrat mensuel résilient plus, mais cela ne prouve pas qu'imposer un contrat plus long réduirait le churn. Ces clients peuvent simplement être moins engagés dès le départ.
- **Variables liées entre elles** : `TotalCharges` dépend de `tenure` et de `MonthlyCharges`, et `nb_services` dépend des services souscrits. Cela explique les coefficients surprenants de la régression logistique (section 8.4).
- **Aucun test statistique** : les différences entre groupes ont été observées et décrites, mais pas testées (par exemple avec un test du χ²). On ne peut donc pas dire si elles sont significatives.

### Limites de la modélisation
Les performances ont été mesurées sur une seule découpe train/test (environ 1 400 clients), donc les écarts de 1 à 2 points entre les modèles ne sont pas fiables. La précision sur la classe Churn est d'environ 51 % : une alerte sur deux est une fausse alerte. Le seuil de décision (0,5) n'a pas été optimisé, et le dataset ne contient pas le coût d'une offre de rétention ni la valeur d'un client conservé, nécessaires pour le choisir. Enfin, les hyperparamètres n'ont pas été ajustés et le modèle n'a pas été testé sur de nouvelles données.

### Pistes d'amélioration
Ajouter des tests statistiques, utiliser une validation croisée, choisir le seuil de décision à partir des coûts réels, et tester le modèle sur des données plus récentes ou sur un autre opérateur.

## 11. Conclusion

Cette analyse de 7 043 clients répond à la question posée : le churn dépend surtout de l'engagement contractuel, de l'ancienneté, du type de connexion et de l'accompagnement du client. Le segment le plus à risque est celui des clients en contrat mensuel avec fibre optique (2 128 clients, 54,6 % de churn). Les deux modèles testés (régression logistique et Random Forest) confirment ces facteurs et détectent environ 78 % des clients qui partent. Nous retenons la régression logistique, plus simple à interpréter.

Ces résultats permettent de cibler les actions de rétention : proposer des contrats plus longs, suivre les nouveaux clients et promouvoir les services de protection. Il faut toutefois rester prudent : les données sont fictives, sans dimension temporelle, et montrent des associations, pas des causes.

**Travaux futurs** : Choisir le seuil de décision selon les coûts réels d'une offre de rétention, et tester le modèle sur des données plus récentes.

## 12. Checklist de reproductibilité

- [x] Le notebook s'exécute de bout en bout sans erreur
- [x] Graine aléatoire fixée (`SEED = 42`, utilisée dans `train_test_split` et les modèles)
- [x] Bibliothèques documentées (voir `requirements.txt`)
- [x] Organisation claire : sections numérotées, code et commentaires associés
- [ ] Fichier de données `WA_Fn-UseC_-Telco-Customer-Churn.csv` placé au même endroit que le notebook (`DATA_PATH` est un chemin relatif)

**Hypothèses et étapes manuelles** : le CSV doit être téléchargé depuis Kaggle (lien en début de notebook).